In [9]:
from Bio import SeqIO
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error as rmse
from scipy.stats import pearsonr


models = [
    'PRIME',
    'DeepSTABp',
    'TemBERTure',
    'TemStaPro',
    'ThermoFormer',
    'TmProt',
    'FINE_650M_FULL_MELTOME',
    'FINE_650M_FLIP',
    'FINE_650M_NO_INT',
]

groups = [
    '1_TEST',
    '3_REVERSED',
    '4_SHUFFLED',
    '2_FIRST_100',
]


file = SeqIO.parse('./data/big_tpp.fasta', "fasta")
big_fasta_labels = dict()

for rec in file:
    d = {_n.split('=')[0]: _n.split('=')[-1] for _n in rec.name.split(';')}
    d['tm'] = float(d['tm'])
    d['seq'] = str(rec._seq)  # remove str(..) if you want access to SeqIO.Seq methods
    big_fasta_labels[d['protein_id']] = d

In [10]:
# Example:

big_fasta_labels['A0A061ACF5_fbl-1']

{'protein_id': 'A0A061ACF5_fbl-1',
 'up_id': 'A0A061ACF5',
 'mapped_id': 'A0A061ACF5',
 'mapped_parcID': 'NaN',
 'sp': 'C.elegans',
 'set': 'Meltome',
 'split': 'train',
 'tm': 49.459,
 'seq': 'MRICFLLLAFLVAETFANELTRCCAGGTRHFKNSNTCSSIKSEGTSMTCQRAASICCLRSLLDNACDSGTDIAKEEESCPSNINILGGGLKKECCDCCLLAKDLLNRNEPCVAPVGFSAGCLRSFNKCCNGDIEITHASEIITGRPLNDPHVLHLGDRCASSHCEHLCHDRGGEKVECSCRSGFDLAPDGMACVDIDECATLMDDCLESQRCLNTPGSFKCIRTLSCGTGYAMDSETERCRDVDECNLGSHDCGPLYQCRNTQGSYRCDAKKCGDGELQNPMTGECTSITCPNGYYPKNGMCNDIDECVTGHNCGAGEECVNTPGSFRCQQKGNLCAHGYEVNGATGFCEDVNECTTGIAACEQKCVNIPGSYQCICDRGFALGPDGTKCEDIDECSIWAGSGNDLCMGGCINTKGSYLCQCPPGYKIQPDGRTCVDVDECAMGECAGSDKVCVNTLGSFKCHSIDCPTNYIHDSLNKNQIADGYSCIKVCSTEDTECLGNHTREVLYQFRAVPSLKTIISPIEVSRIVTHMGVPFSVDYNLDYVGQRHFRIVQERNIGIVQLVKPISGPTVETIKVNIHTKSRTGVILAFNEAIIEISVSKYPF'}

In [13]:
# check results

log_NA = False  # Only for TmProt, 25 sequences do not pass conditions proper to the predictor (sequence size, 'X' amino acids, ...)

for mod in models:
    print(mod)
    for group in groups:
        result = pd.read_csv(f'./results/{mod}/{group}.csv') # fetch result file
        if log_NA:
            NAs = result[result['pred'].isna()] # check for NAs (only TmProt)
            print(f'\tWarning: {len(NAs)} NaN in {mod} ({group}): {NAs['name'].tolist()}')
        result = result.dropna()
        predictions = result['pred']
        labels = result['exp_tm']

        # make dict for easy acces
        name_2_label = result.set_index('name').to_dict()['exp_tm']
        name_2_pred = result.set_index('name').to_dict()['pred']
        assert all(name_2_label[n] == big_fasta_labels[n]['tm'] for n in result['name'])

        # print predictions per group
        corr, p = pearsonr(predictions, labels)
        n_rmse = rmse(predictions, labels) / np.std(labels)
        print(f"\t{group:12s}: pearson = {str(round(corr, 3)):5s}, n_rmse = {str(round(n_rmse, 3)):5s}")
    print()

PRIME
	1_TEST      : pearson = 0.839, n_rmse = 1.616
	3_REVERSED  : pearson = 0.576, n_rmse = 2.178
	4_SHUFFLED  : pearson = 0.364, n_rmse = 2.43 
	2_FIRST_100 : pearson = 0.729, n_rmse = 1.82 

DeepSTABp
	1_TEST      : pearson = 0.758, n_rmse = 0.71 
	3_REVERSED  : pearson = 0.677, n_rmse = 0.905
	4_SHUFFLED  : pearson = 0.519, n_rmse = 1.009
	2_FIRST_100 : pearson = 0.723, n_rmse = 0.703

TemBERTure
	1_TEST      : pearson = 0.84 , n_rmse = 0.558
	3_REVERSED  : pearson = 0.722, n_rmse = 0.773
	4_SHUFFLED  : pearson = 0.637, n_rmse = 0.917
	2_FIRST_100 : pearson = 0.758, n_rmse = 0.662

TemStaPro
	1_TEST      : pearson = 0.676, n_rmse = 1.336
	3_REVERSED  : pearson = 0.48 , n_rmse = 1.54 
	4_SHUFFLED  : pearson = 0.303, n_rmse = 1.628
	2_FIRST_100 : pearson = 0.388, n_rmse = 1.422

ThermoFormer
	1_TEST      : pearson = 0.84 , n_rmse = 0.575
	3_REVERSED  : pearson = 0.738, n_rmse = 0.818
	4_SHUFFLED  : pearson = 0.673, n_rmse = 0.922
	2_FIRST_100 : pearson = 0.762, n_rmse = 0.649

TmPro